# Eval Overview - Combination Perturbation

A lot of the data we train on has a single perturbation applied, but we do have a dataset that has multiple perturbations. Because of this, and future datasets we add, we've built BioJEPA-AC to handle up to 4 applied perturbations. Most of the evaluations we do focus on filtering out these multi-perturbation samples because they confound the signal (is a perturbation that targets gene A substantially different enough from a perturbation that targets gene A and B and should we penalize if it's not?) Because of this we evaluate multi-perturbations separately. 

Combination perturbations is a set of benchmarks that evaluate how well the model can predict the impact of multiple perturbations. The reason to evaluate these is that knocking out two genes in the same pathway can have a different effect than the sum of the individual knockouts. These genetic interactions drive real phenomena like drug synergy and synthetic lethality. This eval uses the gene expression produced by the [linear expression decoder](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_decoders.ipynb). 

Our eval suite does 3 categories of analysis: expression prediction evaluation, additive baseline evaluation, and genetic interaction evaluation. Since we only have 1 dataset in our eval with multiple perturbations, we don't do a per-dataset split. As you walk through the notebook you'll see that we evaluate on multiple dimensions to ensure we have a thorough understanding on where our model is working well and where it's struggling.

In [1]:
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

In [2]:
SEED = 1337
np.random.seed(SEED)

## Data Prep

We'll start by preparing our data. For this evaluation, we need the expression delta both for the combination perturbation cells, and for cells that have single perturbations on each gene. We'll mock up 4 unique combination perturbations that mix targeting 4 unique genes so we'll also have 4 single perturbation samples. 

Since our other explainer notebooks have walked through how sample-level deltas are computed and turned into per-perturbation deltas, we'll stage per-perturbation aggregated deltas directly. We'll create 4 combo perturbations, each combining two genes, along with the single-gene reference data and genetic interaction labels needed for the additive baseline and subtype analysis.

In [3]:
num_genes = 8
num_combos = 4
num_singles = 4
num_samples = 6

**Perturbations**

We'll first start with our perturbations. What's unique here is that we'll stage combination perturbations. Recall that to define a unique perturbation, it's not just about what we target, but the context of it. For a combination perturbation, we expect that there will be more than one unique target meaning a unique sequence and potentially even a unique mode/modality, but, since the perturbations are applied on the same sample, the cell type is the same. Because of this you'll see that we store an array of the combined perturbations as a *composite key* from each perturbation slot. Each slot becomes either $\text{(s or t, seq id or targ id, modality id, mode id)}$. We sort the slots so that the key for A+B and B+A is identical, then wrap them as a tuple alongside the cell type. This gives us a unique, order-independent key for each combination. Note that the assumption is multiple perturbations are applied together, not sequentially, so ordering is ignored. 

We'll create 4 combo perturbations based on CRISPRi perturbations with only targets for simplicity. This means that the target_id represents the gene ID. We'll also create the combination key to represent each combination, and a list of unique genes that correspond with the target ids.

In [4]:
combo_keys = [
    ((('t', 0, 0, 0), ('t', 1, 0, 0)), 0),  # GENE_1 + GENE_2, CRISPRi, cell type 0
    ((('t', 0, 0, 0), ('t', 2, 0, 0)), 0),  # GENE_1 + GENE_3, CRISPRi, cell type 0
    ((('t', 1, 0, 0), ('t', 3, 0, 0)), 0),  # GENE_2 + GENE_4, CRISPRi, cell type 0
    ((('t', 2, 0, 0), ('t', 3, 0, 0)), 0),  # GENE_3 + GENE_4, CRISPRi, cell type 0
]

comp_key_idx = {
    combo_keys[0]: 10,  # GENE_1+GENE_2 combo guide
    combo_keys[1]: 12,  # GENE_1+GENE_3 combo guide
    combo_keys[2]: 14,  # GENE_2+GENE_4 combo guide
    combo_keys[3]: 16,  # GENE_3+GENE_4 combo guide
}

sample_to_pert = [10, 10, 12, 12, 14, 16]

single_gene_names = ['GENE_1', 'GENE_2', 'GENE_3', 'GENE_4']

combo_keys, comp_key_idx, single_gene_names, sample_to_pert

([((('t', 0, 0, 0), ('t', 1, 0, 0)), 0),
  ((('t', 0, 0, 0), ('t', 2, 0, 0)), 0),
  ((('t', 1, 0, 0), ('t', 3, 0, 0)), 0),
  ((('t', 2, 0, 0), ('t', 3, 0, 0)), 0)],
 {((('t', 0, 0, 0), ('t', 1, 0, 0)), 0): 10,
  ((('t', 0, 0, 0), ('t', 2, 0, 0)), 0): 12,
  ((('t', 1, 0, 0), ('t', 3, 0, 0)), 0): 14,
  ((('t', 2, 0, 0), ('t', 3, 0, 0)), 0): 16},
 ['GENE_1', 'GENE_2', 'GENE_3', 'GENE_4'],
 [10, 10, 12, 12, 14, 16])

**Single-Perturbation Real Deltas**

We need a baseline so that we can compare our multi-perturbation expression deltas against. For this we get the per-perturbation mean of the real change in expression. The real delta is calculated as $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$ and then we take the mean across all samples in the perturbation to generate our *real delta*.

Since each single perturbation row corresponds to the gene at the same position in `single_gene_names`, we won't build a separate mapping and will just index directly.

In [5]:
single_pert_real_deltas = np.array([
    [0.1, 0.5, -0.1, 0.8, 0.0, 0.0, 0.0, 1.1],
    [-0.1, 0.0, 1.0, 0.5, 0.3, 0.15, -0.005, -1.25],
    [0.0, 2.0, 1.5, -0.4, 1.0, 0.5, 0.0, 0.0],
    [0.2, -0.1, 0.0, 0.8, -1.0, 0.0, 1.0, 0.5]
])
single_pert_real_deltas.shape, single_pert_real_deltas

((4, 8),
 array([[ 0.1  ,  0.5  , -0.1  ,  0.8  ,  0.   ,  0.   ,  0.   ,  1.1  ],
        [-0.1  ,  0.   ,  1.   ,  0.5  ,  0.3  ,  0.15 , -0.005, -1.25 ],
        [ 0.   ,  2.   ,  1.5  , -0.4  ,  1.   ,  0.5  ,  0.   ,  0.   ],
        [ 0.2  , -0.1  ,  0.   ,  0.8  , -1.   ,  0.   ,  1.   ,  0.5  ]]))

### Multi-Perturbation Samples

Now we'll start building our multi-perturbation sample data. We'll stage sample-level deltas first, then aggregate them to per-perturbation means in a later cell.

**Multi-Perturbation Gene Mapping**

Our first step is to create a link that highlights the specific genes targeted in each perturbation. We need to know which genes are targeted so that in our analysis we can compare the single targets added together against the multi-targeted perturbation. 

Typically we'd scan the perturbation and our cached dictionaries to build out this key. Here we'll just stage it directly.

In [6]:
combo_mapping = {
    '10': ['GENE_1', 'GENE_2'],
    '12': ['GENE_1', 'GENE_3'],
    '14': ['GENE_2', 'GENE_4'],
    '16': ['GENE_3', 'GENE_4'],
}
combo_mapping

{'10': ['GENE_1', 'GENE_2'],
 '12': ['GENE_1', 'GENE_3'],
 '14': ['GENE_2', 'GENE_4'],
 '16': ['GENE_3', 'GENE_4']}

In [7]:
combo_to_genes = {}
for key in combo_keys:
    sid = comp_key_idx.get(key)
    genes = combo_mapping.get(str(sid)) if sid is not None else None
    combo_to_genes[key] = tuple(genes) if genes else None
    print(f'Key {combo_keys.index(key)}: first_seq_idx={sid} -> combo_mapping["{sid}"] -> {combo_to_genes[key]}')

combo_to_genes

Key 0: first_seq_idx=10 -> combo_mapping["10"] -> ('GENE_1', 'GENE_2')
Key 1: first_seq_idx=12 -> combo_mapping["12"] -> ('GENE_1', 'GENE_3')
Key 2: first_seq_idx=14 -> combo_mapping["14"] -> ('GENE_2', 'GENE_4')
Key 3: first_seq_idx=16 -> combo_mapping["16"] -> ('GENE_3', 'GENE_4')


{((('t', 0, 0, 0), ('t', 1, 0, 0)), 0): ('GENE_1', 'GENE_2'),
 ((('t', 0, 0, 0), ('t', 2, 0, 0)), 0): ('GENE_1', 'GENE_3'),
 ((('t', 1, 0, 0), ('t', 3, 0, 0)), 0): ('GENE_2', 'GENE_4'),
 ((('t', 2, 0, 0), ('t', 3, 0, 0)), 0): ('GENE_3', 'GENE_4')}

**Multi-Pert Per-Sample Deltas**

Now we'll stage the per-sample expression deltas for our multi-perturbation samples. We need the change in expression for both the predicted and real values:
1. `multi_pred_deltas` - the predicted change in expression as calculated by $\hat{\delta}_g = \hat{x}^{\text{case}}_g - \hat{x}^{\text{ctrl}}_g$. This value compares the predicted perturbed expression (`pred_case`) from the predicted control expression (`pred_control`). We use the predicted control expression to isolate BioJEPA-AC's learned perturbation effect from any baseline reconstruction error.
2. `multi_real_deltas` - the real change in expression as calculated by $\delta_g = x^{\text{case}}_g - x^{\text{ctrl}}_g$. This is our source of truth.

Recall that our first two samples link to the first pert, our next two to the second combination pert, and then the final two each to their own unique pert. With these deltas, we've staged the data so that the additive baseline is closer to reality for perts 0 and 2, while the model does better on perts 1 and 3. You'll see how this gets shown in different calculations.

In [8]:
multi_real_deltas = np.array([
    [0.55, 0.55, 0.95, 1.3, 0.35, 0.2, 0.45, -0.1],
    [0.45, 0.45, 0.85, 1.2, 0.25, 0.1, 0.35, -0.2],
    [0.15, 0.15, 1.45, 0.45, 1.05, 0.55, 0.05, -0.05],
    [0.05, 0.05, 1.35, 0.35, 0.95, 0.45, -0.05, -0.15],
    [0.1, -0.1, 0.5, 0.6, -0.7, 0.15, 0.995, -0.75],
    [0.2, 1.9, 1.5, 0.4, 0.8, -0.2, 1.0, 0.5],
])
multi_real_deltas.shape, multi_real_deltas

((6, 8),
 array([[ 0.55 ,  0.55 ,  0.95 ,  1.3  ,  0.35 ,  0.2  ,  0.45 , -0.1  ],
        [ 0.45 ,  0.45 ,  0.85 ,  1.2  ,  0.25 ,  0.1  ,  0.35 , -0.2  ],
        [ 0.15 ,  0.15 ,  1.45 ,  0.45 ,  1.05 ,  0.55 ,  0.05 , -0.05 ],
        [ 0.05 ,  0.05 ,  1.35 ,  0.35 ,  0.95 ,  0.45 , -0.05 , -0.15 ],
        [ 0.1  , -0.1  ,  0.5  ,  0.6  , -0.7  ,  0.15 ,  0.995, -0.75 ],
        [ 0.2  ,  1.9  ,  1.5  ,  0.4  ,  0.8  , -0.2  ,  1.   ,  0.5  ]]))

In [9]:
multi_pred_deltas = np.array([
    [0.85, 0.25, 1.25, 0.95, 0.05, 0.45, 0.25, 0.15],
    [0.75, 0.15, 1.15, 0.85, -0.05, 0.35, 0.15, 0.05],
    [0.2, 0.2, 1.4, 0.4, 1.0, 0.5, 0.1, 0.0],
    [0.1, 0.1, 1.3, 0.3, 0.9, 0.4, 0.0, -0.1],
    [0.5, 0.3, 0.55, 0.65, -0.3, -0.2, 1.3, -0.35],
    [0.25, 1.85, 1.45, 0.35, 0.75, -0.15, 0.95, 0.45],
])
multi_pred_deltas.shape, multi_pred_deltas

((6, 8),
 array([[ 0.85,  0.25,  1.25,  0.95,  0.05,  0.45,  0.25,  0.15],
        [ 0.75,  0.15,  1.15,  0.85, -0.05,  0.35,  0.15,  0.05],
        [ 0.2 ,  0.2 ,  1.4 ,  0.4 ,  1.  ,  0.5 ,  0.1 ,  0.  ],
        [ 0.1 ,  0.1 ,  1.3 ,  0.3 ,  0.9 ,  0.4 ,  0.  , -0.1 ],
        [ 0.5 ,  0.3 ,  0.55,  0.65, -0.3 , -0.2 ,  1.3 , -0.35],
        [ 0.25,  1.85,  1.45,  0.35,  0.75, -0.15,  0.95,  0.45]]))

## Sample-Level Metrics

As part of our combination-perturbation eval, we report out two sample level metrics: Mean-Squared-Error (MSE) and Pearson correlation. These sample level metrics are the same values we calculate in the [Gene Expression Prediction evals](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_expr_prediction.ipynb), just with a filter to average only over samples with multiple perturbations.

### Mean Squared Error (MSE)

Our first sample calculation will evaluate simply how far off are our expression change predictions from the real changes. To do this we calculate the mean squared error as follows:

$$\text{MSE}_{\text{sample}}=\frac{1}{N}\sum_{i=1}^{N}\frac{1}{G}\sum_{g=1}^{G}(\hat{\delta}_{i,g} - \delta_{i,g})^2$$

If you look at the formula, you can see that this calculation will quickly be dominated by the largest expression changes. This is a large reason why we log normalize expression counts, that way large order of magnitude changes do not dominate our optimization. This is especially important since in many of our perturbation cases a vast majority of our genes will have very minor changes and we need to make sure we're able to predict that just as well as large changes.

After we calculate per sample MSE, we then take a final mean to get the dataset level mean MSE. We can see that our first, second, and fifth samples all dominate our MSE calculation, raising the mean.

In [10]:
per_sample_mse = np.mean((multi_pred_deltas - multi_real_deltas)**2, axis=1)
per_sample_mse.shape, per_sample_mse

((6,),
 array([0.0809375 , 0.0809375 , 0.0025    , 0.0025    , 0.10756563,
        0.0025    ]))

In [11]:
sample_mse = np.mean(per_sample_mse)
sample_mse

np.float64(0.04615677083333333)

### Top 20 DEG Pearson's $r$

We next calculate the Pearson's correlation coefficient, $r$, per sample. Pearson's $r$ measures how linearly correlated the predicted and real expression delta profiles are across genes for each sample, regardless of scale. The value of this eval is that if our expression is very linearly correlated, but the values are off, our model is still useful as it's learned the profile and we just need to find the right scaling factor. For our sample level Pearson's, since this is a benchmark that we compare against other models, we calculate per sample but restrict to the top 20 differentially expressed genes. We pull out the largest (by absolute value) top 20 gene expression changes from our real values `real_delta`, and then compare those against the calculated changes for those same genes and then take the dataset mean. The resulting calculation is:

$$r_{\text{sample}} = \frac{1}{N}\sum_{i=1}^{N}\frac{\sum_{k=1}^{K}(\hat{\delta}_{i,k} - \bar{\hat{\delta}}_i)(\delta_{i,k} - \bar{\delta}_i)}{\sqrt{\sum_{k=1}^{K}(\hat{\delta}_{i,k} - \bar{\hat{\delta}}_i)^{2}} \cdot \sqrt{\sum_{k=1}^{K}(\delta_{i,k} - \bar{\delta}_i)^{2}}}$$

where $K=20$ genes are selected per sample as the largest $|\delta_{i,g}|$. For this example, since we only have 8 genes, instead of taking the top 20, we'll show it by taking only the top 3. You'll see that because of how we staged our data, the first two samples have lower correlation while the last four have high correlation.

In [12]:
TOP_K = 3
per_sample_corr = []

In [13]:
for i in range(num_samples):
    print(f'----CELL {i}----')
    top_20_idx = np.argsort(np.abs(multi_real_deltas[i]))[-TOP_K:]
    pred_top, real_top = multi_pred_deltas[i][top_20_idx], multi_real_deltas[i][top_20_idx]
    print(f'Top {TOP_K} expression deltas: pred {pred_top} | real {real_top}')
    corr, _ = pearsonr(pred_top, real_top)
    p_corr = 0.0 if np.isnan(corr) else float(corr)
    print(f'Pearson\'s r {p_corr}')
    per_sample_corr.append(p_corr)

per_sample_corr = np.array(per_sample_corr)
per_sample_corr.shape, per_sample_corr

----CELL 0----
Top 3 expression deltas: pred [0.25 1.25 0.95] | real [0.55 0.95 1.3 ]
Pearson's r 0.7096708298148764
----CELL 1----
Top 3 expression deltas: pred [0.15 1.15 0.85] | real [0.45 0.85 1.2 ]
Pearson's r 0.7096708298148766
----CELL 2----
Top 3 expression deltas: pred [0.5 1.  1.4] | real [0.55 1.05 1.45]
Pearson's r 0.9999999999999999
----CELL 3----
Top 3 expression deltas: pred [0.4 0.9 1.3] | real [0.45 0.95 1.35]
Pearson's r 1.0
----CELL 4----
Top 3 expression deltas: pred [-0.3  -0.35  1.3 ] | real [-0.7   -0.75   0.995]
Pearson's r 0.9999989183875729
----CELL 5----
Top 3 expression deltas: pred [0.95 1.45 1.85] | real [1.  1.5 1.9]
Pearson's r 1.0


((6,),
 array([0.70967083, 0.70967083, 1.        , 1.        , 0.99999892,
        1.        ]))

In [14]:
sample_corr = np.mean(per_sample_corr)
sample_corr

np.float64(0.9032234296695543)

## Perturbation Level Expression Eval

Our next set of evaluations is to similarly run our "per-perturbation" level expression analysis, just in this case focusing on "per-multi-pert". Since these analyses are the same as our [Gene Expression Analysis](https://github.com/GPTomics/biojepa/blob/main/layer_explainers/explainer_eval_expr_prediction.ipynb) we won't reshow the math, we simply encourage you to review the other notebook to understand how the calculation is done. As part of the analysis we calculate: $R^2$ on all genes and top 50 DEGs, MSE, Pearson on all genes, Pearson on expression deltas, Pearson on top DEGs, Centroid Accuracy, Versus Baseline, Severity, and magnitude-based error analysis.

By calculating the same statistics we can compare the single-pert statistics with the multi-pert and see how different the model performance is at each category. 

We will, however, still aggregate our expression delta per perturbation as it will be important for our other analyses.

### Aggregate Data Per Perturbation

We'll first create our per-perturbation data. We'll use our `sample_to_pert` to help identify which perturbation each sample belongs to. We'll iterate through our perturbations, find which samples belong to the perturbation, pluck out the expression data for the samples, and then run a mean across them to get a single value per perturbation.

In [15]:
multi_pert_pred_delta = np.zeros((num_combos, num_genes))
multi_pert_real_delta = np.zeros((num_combos, num_genes))

In [16]:
for i, p in enumerate(combo_mapping.keys()):
    mask = [s == int(p) for s in sample_to_pert]
    multi_pert_pred_delta[i] = multi_pred_deltas[mask].mean(axis=0)
    multi_pert_real_delta[i] = multi_real_deltas[mask].mean(axis=0)

multi_pert_pred_delta.shape, multi_pert_pred_delta, multi_pert_real_delta

((4, 8),
 array([[ 0.8 ,  0.2 ,  1.2 ,  0.9 ,  0.  ,  0.4 ,  0.2 ,  0.1 ],
        [ 0.15,  0.15,  1.35,  0.35,  0.95,  0.45,  0.05, -0.05],
        [ 0.5 ,  0.3 ,  0.55,  0.65, -0.3 , -0.2 ,  1.3 , -0.35],
        [ 0.25,  1.85,  1.45,  0.35,  0.75, -0.15,  0.95,  0.45]]),
 array([[ 0.5  ,  0.5  ,  0.9  ,  1.25 ,  0.3  ,  0.15 ,  0.4  , -0.15 ],
        [ 0.1  ,  0.1  ,  1.4  ,  0.4  ,  1.   ,  0.5  ,  0.   , -0.1  ],
        [ 0.1  , -0.1  ,  0.5  ,  0.6  , -0.7  ,  0.15 ,  0.995, -0.75 ],
        [ 0.2  ,  1.9  ,  1.5  ,  0.4  ,  0.8  , -0.2  ,  1.   ,  0.5  ]]))

## Additive Baseline

Our first unique evaluation for combination perturbation is the additive baseline. These evaluations run aggregated "per-perturbation". In this evaluation, we compare the real change in expression with both the predicted change from the combination perturbation and the sum of the real change for each single perturbation. The goal is to determine: is our model actually better than just taking the sum of two known impacts. While many multi-perturbations may be roughly additive, the goal is that the model learns when it's not. Note that we do run this across all genes, so, in the production eval, we will see a lot of the correlations driven by how well the genes that rarely change are predicted. 

We'll first start by calculating our *additive deltas*.

**Calculate Additive Deltas**

For this evaluation, we have ensured that we are only looking at combination perturbations where we have samples that represent each individual perturbation also. Since we already know the expression delta for the combination perturbation samples, we now have to create the additive samples. We do this by splitting a combination perturbation by each unique perturbation, and then summing the expression delta for each part. We calculate the additive delta as:
$$
\delta^{\text{add}}_g = \delta^{\text{single\_A}}_g + \delta^{\text{single\_B}}_g
$$

We'll print out the loop so that you can see how the addition can in some cases subtract while in others add. With this, if we think about it biologically, we can start to see where genetic interactions can come into play. For example, if one of the two perturbations fully knocks out a network of expression, and the other is a slight enhancer, the knockout might dominate.

In [17]:
gene_name_to_idx = {g: i for i, g in enumerate(single_gene_names)}
additive_deltas = {}
gene_name_to_idx

{'GENE_1': 0, 'GENE_2': 1, 'GENE_3': 2, 'GENE_4': 3}

In [18]:
for key in combo_keys:
    print(f'---- Pert {key} ----')
    genes = combo_to_genes[key]

    delta = []
    for g in genes:
        idx = gene_name_to_idx[g]
        pert_delt = single_pert_real_deltas[idx]
        print(f'gene {g}: {pert_delt}')
        delta.append(pert_delt)
    additive = np.sum(delta, axis=0)
    additive_deltas[key] = additive
    print(f'sum: {additive}')

additive_deltas

---- Pert ((('t', 0, 0, 0), ('t', 1, 0, 0)), 0) ----
gene GENE_1: [ 0.1  0.5 -0.1  0.8  0.   0.   0.   1.1]
gene GENE_2: [-0.1    0.     1.     0.5    0.3    0.15  -0.005 -1.25 ]
sum: [ 0.     0.5    0.9    1.3    0.3    0.15  -0.005 -0.15 ]
---- Pert ((('t', 0, 0, 0), ('t', 2, 0, 0)), 0) ----
gene GENE_1: [ 0.1  0.5 -0.1  0.8  0.   0.   0.   1.1]
gene GENE_3: [ 0.   2.   1.5 -0.4  1.   0.5  0.   0. ]
sum: [0.1 2.5 1.4 0.4 1.  0.5 0.  1.1]
---- Pert ((('t', 1, 0, 0), ('t', 3, 0, 0)), 0) ----
gene GENE_2: [-0.1    0.     1.     0.5    0.3    0.15  -0.005 -1.25 ]
gene GENE_4: [ 0.2 -0.1  0.   0.8 -1.   0.   1.   0.5]
sum: [ 0.1   -0.1    1.     1.3   -0.7    0.15   0.995 -0.75 ]
---- Pert ((('t', 2, 0, 0), ('t', 3, 0, 0)), 0) ----
gene GENE_3: [ 0.   2.   1.5 -0.4  1.   0.5  0.   0. ]
gene GENE_4: [ 0.2 -0.1  0.   0.8 -1.   0.   1.   0.5]
sum: [0.2 1.9 1.5 0.4 0.  0.5 1.  0.5]


{((('t', 0, 0, 0), ('t', 1, 0, 0)),
  0): array([ 0.   ,  0.5  ,  0.9  ,  1.3  ,  0.3  ,  0.15 , -0.005, -0.15 ]),
 ((('t', 0, 0, 0), ('t', 2, 0, 0)),
  0): array([0.1, 2.5, 1.4, 0.4, 1. , 0.5, 0. , 1.1]),
 ((('t', 1, 0, 0), ('t', 3, 0, 0)),
  0): array([ 0.1  , -0.1  ,  1.   ,  1.3  , -0.7  ,  0.15 ,  0.995, -0.75 ]),
 ((('t', 2, 0, 0), ('t', 3, 0, 0)),
  0): array([0.2, 1.9, 1.5, 0.4, 0. , 0.5, 1. , 0.5])}

### Mean Squared Error (MSE) Comparison

Now we are ready to compare both the accuracy of our predicted multi-perturbation impact against summing known impacts of each single perturbation (aka single gene real pert delta). For each comparison against the real combination expression delta, we compute the Mean Squared Error (MSE) as:
$$\begin{aligned}
\text{MSE}_{\text{pred}} &= \frac{1}{G}\sum_{g=1}^{G}(\hat{\delta}_g - \delta_g)^2 \\
\text{MSE}_{\text{additive}} &= \frac{1}{G}\sum_{g=1}^{G}(\delta^{\text{add}}_g - \delta_g)^2
\end{aligned}$$

The model isn't expected to always beat additive, especially when the combination perturbation is mostly additive since we're trying to beat lab-validated additive impact. When, however, there are genetic interactions, we hope the model is able to beat simply summing the single gene deltas.

We'll start by calculating per-perturbation MSE for both comparisons. Then we'll calculate how often our model is better, and the mean of each comparison's MSE.

In [19]:
per_key_results = {}
pred_mses_list = []
additive_mses_list = []

In [20]:
for i, key in enumerate(combo_keys):
    print(f'---- Pert {key} ----')
    genes = combo_to_genes[key]
    real = multi_pert_real_delta[i]
    pred = multi_pert_pred_delta[i]
    additive = additive_deltas[key]

    pred_mse = float(np.mean((pred - real) ** 2))
    additive_mse = float(np.mean((additive - real) ** 2))
    pred_mses_list.append(pred_mse)
    additive_mses_list.append(additive_mse)

    print(f'MSEs: Predicted {pred_mse} | Additive {additive_mse}')

    per_key_results[key] = {'genes': genes, 'pred_mse': pred_mse, 'additive_mse': additive_mse, 'additive_delta': additive}

---- Pert ((('t', 0, 0, 0), ('t', 1, 0, 0)), 0) ----
MSEs: Predicted 0.08093750000000001 | Additive 0.052065625000000004
---- Pert ((('t', 0, 0, 0), ('t', 2, 0, 0)), 0) ----
MSEs: Predicted 0.0024999999999999996 | Additive 0.9
---- Pert ((('t', 1, 0, 0), ('t', 3, 0, 0)), 0) ----
MSEs: Predicted 0.10756562500000001 | Additive 0.09250000000000001
---- Pert ((('t', 2, 0, 0), ('t', 3, 0, 0)), 0) ----
MSEs: Predicted 0.0024999999999999996 | Additive 0.14125000000000001


**Model Beat Rate**

Now we're ready to determine how often our model is better than the simple additive default. We calculate it as:
$$
\text{beat\_rate} = \frac{1}{C}\sum_{c=1}^{C}\mathbf{1}[\text{MSE}^{\text{model}}_c < \text{MSE}^{\text{add}}_c]
$$
Because of how we staged the data, you'll see the model wins on the second and fourth perturbation (perts 1 and 3), where the real combo effect deviates significantly from additive. The additive baseline wins on the first and third where reality is closer to the sum of singles.

In [21]:
wins = 0
for i in range(num_combos):
    pred = pred_mses_list[i]
    add = additive_mses_list[i]
    win = pred < add
    wins += win
    print(f'Pert {i}: {'win' if win else 'loss'}')

model_beats_additive_rate = float(wins / num_combos)
model_beats_additive_rate

Pert 0: loss
Pert 1: win
Pert 2: loss
Pert 3: win


0.5

**MSE Average** 

Now that we have the win rate, we can see how, across the full dataset, our MSE averages compare. Even if our win rate for the prediction is not perfect, the average MSE can tell us how across all the multi-pert samples we're doing. What we'll see here is that the model has a lower average MSE, mainly because the additive baseline's large errors on perts 1 and 3 dominate the average.

In [22]:
pred_mse = float(np.mean(pred_mses_list))
pred_mse

0.048375781250000006

In [23]:
additive_mse = float(np.mean(additive_mses_list))
additive_mse

0.29645390625

### Pearson Correlation Comparison

Next we compare how well correlated our predictions are across all the genes. We use Pearson correlation to measure the linear correlation of both our predicted and additive delta against the real delta as:
$$
\text{r} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2} \sqrt{\sum_{i=1}^{n}(y_i - \bar{y})^2}}
$$

Pearson ignores magnitude so this is where the minor changes in genes really impacts the value. You'll see that the model is better correlated than additive on perts 1 and 3, matching the MSE results, while additive is closer on perts 0 and 2.

In [24]:
pred_pearson_list = []
additive_pearson_list = []

In [25]:
for i, key in enumerate(combo_keys):
    print(f'---- Pert {key} ----')
    genes = combo_to_genes[key]
    real = multi_pert_real_delta[i]
    pred = multi_pert_pred_delta[i]
    additive = additive_deltas[key]

    pred_r, _ = pearsonr(pred, real)
    add_r, _ = pearsonr(additive, real)
    pred_pearson_list.append(float(pred_r))
    additive_pearson_list.append(float(add_r))

    print(f'Pearson: Predicted {pred_r} | Additive {add_r}')

---- Pert ((('t', 0, 0, 0), ('t', 1, 0, 0)), 0) ----
Pearson: Predicted 0.755771706506401 | Additive 0.9040512216835425
---- Pert ((('t', 0, 0, 0), ('t', 2, 0, 0)), 0) ----
Pearson: Predicted 0.9979116045457206 | Additive 0.17950736539035678
---- Pert ((('t', 1, 0, 0), ('t', 3, 0, 0)), 0) ----
Pearson: Predicted 0.8961070466362157 | Additive 0.9466729633598918
---- Pert ((('t', 2, 0, 0), ('t', 3, 0, 0)), 0) ----
Pearson: Predicted 0.9986814687403722 | Additive 0.8243809658591821


**Pearson Average** 

Now that we have the per perturbation Pearson, we can take the averages. Because the additive baseline's poor correlation on pert 1 pulls its average down significantly, you'll see the model's mean Pearson ends up higher than the additive's.

In [26]:
model_pearson = float(np.mean(pred_pearson_list))
model_pearson

0.9121179566071773

In [27]:
additive_pearson = float(np.mean(additive_pearson_list))
additive_pearson

0.7136531290732433

### Non-Additive Gene Expression Evaluation

For our non-additive gene evaluation, we want to focus specifically for each combination perturbation on the top N genes that deviate most from the additive effect. This is similar in spirit to a "top DEG" evaluation where we focus only on the strongest changes. We calculate per gene the deviation as:
$$
\text{deviation}_g = |\delta^{\text{real}}_g - \delta^{\text{add}}_g|
$$

We use this devation to identify which genes to include in our non-additive calculateions. In production we focus on the top 20 genes, but since we only have 8 here, we'll focus on the top 3. It's important to remember that this is not the ones with the largest delta, but rather the genes with the largest deviation from the additive effect. You'll see that each perturbation picks a different set of top genes, and that the third perturbation (pert 2) in particular shows a significant improvement in Pearson when restricted to only these non-additive genes.

In [28]:
TOP_N = 3
nonadd_pred_mses = []
nonadd_pearsons = []

In [29]:
for i, key in enumerate(combo_keys):
    print(f'---- Pert {i} ----')
    genes = combo_to_genes[key]
    real = multi_pert_real_delta[i]
    pred = multi_pert_pred_delta[i]
    additive = additive_deltas[key]

    per_gene_dev = np.abs(real - additive)
    top_idx = np.argsort(per_gene_dev)[-TOP_N:]
    print(f'genes with top deviation: {top_idx} | {per_gene_dev}')

    mse = float(np.mean((pred[top_idx] - real[top_idx]) ** 2))
    nonadd_pred_mses.append(mse)

    pearson, _ = pearsonr(pred[top_idx], real[top_idx])
    nonadd_pearsons.append(float(pearson))

    print(f'mse {mse} | pearson {pearson}')

---- Pert 0 ----
genes with top deviation: [3 6 0] | [5.00000000e-01 0.00000000e+00 1.11022302e-16 5.00000000e-02
 0.00000000e+00 2.77555756e-17 4.05000000e-01 1.11022302e-16]
mse 0.08416666666666671 | pearson 0.691733407030543
---- Pert 1 ----
genes with top deviation: [6 7 1] | [0.  2.4 0.  0.  0.  0.  0.  1.2]
mse 0.002500000000000001 | pearson 0.9999999999999998
---- Pert 2 ----
genes with top deviation: [7 2 3] | [0.  0.  0.5 0.7 0.  0.  0.  0. ]
mse 0.055000000000000014 | pearson 0.9997025727391705
---- Pert 3 ----
genes with top deviation: [7 5 4] | [0.  0.  0.  0.  0.8 0.7 0.  0. ]
mse 0.002500000000000002 | pearson 0.9993216505720215


In [30]:
nonadd_model_mse = np.mean(nonadd_pred_mses)
nonadd_model_mse

np.float64(0.03604166666666668)

In [31]:
nonadd_model_pearsons = np.mean(nonadd_pearsons)
nonadd_model_pearsons

np.float64(0.9226894075854337)

## Genetic Interaction Subtypes Analysis

Since we're using combination perturbations, another analysis we can look at is if the subtypes have a known genetic interaction and, if so, does our model do better on certain interactions than others. Examples of interactions include *synergistic* (combo effect exceeds the sum of singles), *alleviating* (combo effect is weaker, genes compensate), *suppressive* (one gene dominates), among others. 

For this analysis we use a known dictionary of subtype interactions to group our combination perturbations. Based on those groupings we then calculate our predicted vs additive MSEs and interaction Pearson correlation.

### Genetic Interaction Subtype Mapping
We'll start by labeling our interaction subtypes by creating a dictionary of genes to genetic interactions and then mapping our perturbations to them. Note that we create a dictionary with all the combinations of genes regardless of order for our subtype matching. 

After we have the keys we then add the subtype to our dictionary that holds the additive delta.

In [32]:
name_to_subtype = {}
labeled_combos = {}
gi_subtypes = {
    'GENE_1_GENE_2': 'Synergistic',
    'GENE_1_GENE_3': 'Alleviating',
    'GENE_3_GENE_4': 'Suppressive',
}
subtypes = set([gi_subtypes[i] for i in gi_subtypes.keys()])
subtypes

{'Alleviating', 'Suppressive', 'Synergistic'}

In [33]:
for combo_name, subtype in gi_subtypes.items():
    name_to_subtype[combo_name] = subtype

for key in combo_keys:
    genes = combo_to_genes[key]
    fwd = f'{genes[0]}_{genes[1]}'
    rev = f'{genes[1]}_{genes[0]}'
    if fwd in name_to_subtype and rev not in name_to_subtype:
        name_to_subtype[rev] = name_to_subtype[fwd]
        
name_to_subtype

{'GENE_1_GENE_2': 'Synergistic',
 'GENE_1_GENE_3': 'Alleviating',
 'GENE_3_GENE_4': 'Suppressive',
 'GENE_2_GENE_1': 'Synergistic',
 'GENE_3_GENE_1': 'Alleviating',
 'GENE_4_GENE_3': 'Suppressive'}

In [34]:
for key, kres in per_key_results.items():
    genes = kres['genes']
    combo_name = f'{genes[0]}_{genes[1]}'
    subtype = name_to_subtype.get(combo_name)
    if subtype:
        labeled_combos[key] = {'genes': genes, 'subtype': subtype, 'additive_delta': kres['additive_delta']}
        print(f'{genes[0]}+{genes[1]}: {subtype}')
    else:
        print(f'{genes[0]}+{genes[1]}: no label')
labeled_combos

GENE_1+GENE_2: Synergistic
GENE_1+GENE_3: Alleviating
GENE_2+GENE_4: no label
GENE_3+GENE_4: Suppressive


{((('t', 0, 0, 0), ('t', 1, 0, 0)), 0): {'genes': ('GENE_1', 'GENE_2'),
  'subtype': 'Synergistic',
  'additive_delta': array([ 0.   ,  0.5  ,  0.9  ,  1.3  ,  0.3  ,  0.15 , -0.005, -0.15 ])},
 ((('t', 0, 0, 0), ('t', 2, 0, 0)), 0): {'genes': ('GENE_1', 'GENE_3'),
  'subtype': 'Alleviating',
  'additive_delta': array([0.1, 2.5, 1.4, 0.4, 1. , 0.5, 0. , 1.1])},
 ((('t', 2, 0, 0), ('t', 3, 0, 0)), 0): {'genes': ('GENE_3', 'GENE_4'),
  'subtype': 'Suppressive',
  'additive_delta': array([0.2, 1.9, 1.5, 0.4, 0. , 0.5, 1. , 0.5])}}

### Interaction Pearson 

Since we know what the additive baseline is, we can then use that knowledge to see how close our predicted change in expression without the additive baseline is to the real change without the additive baseline. We call this the *interaction effect* and we calculate it as:

$$\begin{aligned}
\text{interaction}^{\text{real}}_g &= \delta^{\text{real}}_g - \delta^{\text{add}}_g \\\text{interaction}^{\text{pred}}_g &= \delta^{\text{pred}}_g - \delta^{\text{add}}_g
\end{aligned}$$

After we calculate the predicted and real interaction effect we then want to know how correlated they are. We use the Pearson correlation to see if we can linearly correlate the predicted interaction effect with the real interaction effect. After we calculate them, we then take the mean by subtype so that we can understand what subtypes we do well on and which we don't. Based on how we staged the data, you'll see that the model has high correlation on most subtypes except for Synergistic.

In [35]:
subtype_pearson = {i:[] for i in subtypes}
subtype_pearson

{'Suppressive': [], 'Synergistic': [], 'Alleviating': []}

In [36]:
for key, info in labeled_combos.items():
    i = combo_keys.index(key)
    subtype = info.get('subtype','')
    print(f'---- Pert {i} | {subtype} ----')
    genes = info['genes']
    real = multi_pert_real_delta[i]
    pred = multi_pert_pred_delta[i]
    additive = info['additive_delta']

    interaction_real = real - additive
    interaction_pred = pred - additive

    print(f'real {interaction_real}')
    print(f'pred {interaction_pred}')
    p_int, _ = pearsonr(interaction_pred, interaction_real)

    subtype_pearson[subtype].append(p_int)

---- Pert 0 | Synergistic ----
real [ 5.00000000e-01  0.00000000e+00 -1.11022302e-16 -5.00000000e-02
  0.00000000e+00  2.77555756e-17  4.05000000e-01 -1.11022302e-16]
pred [ 0.8   -0.3    0.3   -0.4   -0.3    0.25   0.205  0.25 ]
---- Pert 1 | Alleviating ----
real [ 0.  -2.4  0.   0.   0.   0.   0.  -1.2]
pred [ 0.05 -2.35 -0.05 -0.05 -0.05 -0.05  0.05 -1.15]
---- Pert 3 | Suppressive ----
real [ 0.   0.   0.   0.   0.8 -0.7  0.   0. ]
pred [ 0.05 -0.05 -0.05 -0.05  0.75 -0.65 -0.05 -0.05]


In [37]:
subtype_pearson_means = {k: np.mean(v) for k, v in subtype_pearson.items()}
subtype_pearson_means

{'Suppressive': np.float64(0.9948084009408477),
 'Synergistic': np.float64(0.6801480743229122),
 'Alleviating': np.float64(0.998644735408801)}

### Per-Subtype MSE 
Our final per-subtype metric we calculate is the same mean-squared-error that we calculate across the combination perturbation: one for the additive compared to the real delta, and one for the predicted compared to the real delta. We use the same calculation, just focused on the perturbations in the subtype. This allows us a deeper understanding of where the model is doing better than additive and where it's worse.

In [38]:
subtype_mse_additive = {i:[] for i in subtypes}
subtype_mse_pred = {i:[] for i in subtypes}
subtype_mse_additive, subtype_mse_pred

({'Suppressive': [], 'Synergistic': [], 'Alleviating': []},
 {'Suppressive': [], 'Synergistic': [], 'Alleviating': []})

In [39]:
for key, info in labeled_combos.items():
    i = combo_keys.index(key)
    subtype = info.get('subtype','')
    print(f'---- Pert {i} | {subtype} ----')
    genes = info['genes']
    real = multi_pert_real_delta[i]
    pred = multi_pert_pred_delta[i]
    additive = info['additive_delta']

    pred_mse = float(np.mean((pred - real) ** 2))
    additive_mse = float(np.mean((additive - real) ** 2))
    
    subtype_mse_pred[subtype].append(pred_mse)
    subtype_mse_additive[subtype].append(additive_mse)
    
    print(f'MSEs: Predicted {pred_mse} | Additive {additive_mse}')

---- Pert 0 | Synergistic ----
MSEs: Predicted 0.08093750000000001 | Additive 0.052065625000000004
---- Pert 1 | Alleviating ----
MSEs: Predicted 0.0024999999999999996 | Additive 0.9
---- Pert 3 | Suppressive ----
MSEs: Predicted 0.0024999999999999996 | Additive 0.14125000000000001


**Mean MSE by Subtype**

Now we're ready to calculate the mean of the MSEs by subtype. As we calculate the mean you'll notice that while we saw the Pearson correlation was quite poor for the synergistic subtype, the MSE is quite low, though the predicted is worse than the additive.

In [40]:
subtype_mse_pred_means = {k: np.mean(v) for k, v in subtype_mse_pred.items()}
subtype_mse_add_means = {k: np.mean(v) for k, v in subtype_mse_additive.items()}
for s in subtypes:
    mse_p = subtype_mse_pred_means[s]
    mse_a = subtype_mse_add_means[s]
    print(f'{s} MSEs: predicted {mse_p} | additive {mse_a}')

Suppressive MSEs: predicted 0.0024999999999999996 | additive 0.14125000000000001
Synergistic MSEs: predicted 0.08093750000000001 | additive 0.052065625000000004
Alleviating MSEs: predicted 0.0024999999999999996 | additive 0.9


## Combination Perturbation Final Wrapup

We've now walked through our evaluation of combination perturbation impacts. While a lot of the evaluations are similar to the gene expression evaluations, you saw how we benchmark more deeply on whether the model has learned any genetic interactions or not as well. As mentioned, we do not run this split up by dataset as we only currently have a single dataset that has multiple perturbations. In the future we will split up by dataset. These combination perturbation evals are another item that helps show how well our model has learned the physics of a cell, though, the dataset is relatively small so we do not yet expect fantastic results.